# 用 Python 函数处理

Polars 可以调用用户函数来转换和处理数据。内置函数和运算符覆盖不到的特定处理，用用户函数补上。本章说明如何在 Polars 中定义和调用用户函数，以及如何借它写出更灵活的处理。

In [1]:
import polars as pl
import numpy as np
from helper.jupyter import row

## pipe

Polars 提供 `pipe` 方法，用来提高代码可读性、灵活套用函数。主要用在两个地方。

1. `DataFrame.pipe()`
2. `Expr.pipe()`

下面分别举例说明。

### DataFrame.pipe

`DataFrame.pipe` 把函数插进链式处理里。复杂的处理可以按步骤写下来，可读性和可维护性都更好。

`pipe()` 的第一个参数是执行的函数，`DataFrame` 本身会自动作为该函数的第一个参数传入。`pipe()` 其余的定位参数和关键字参数原样转给这个函数。

In [2]:
df = pl.DataFrame({
    "A": [1, 2, 3],
    "B": [4, 5, 6],
    "C": [7, 8, 9]
})

def add_column(df, col_name, value):
    return df.with_columns(pl.lit(value).alias(col_name))

def multiply_column(df, col_name, factor):
    return df.with_columns((pl.col(col_name) * factor).alias(col_name))

df_res = (
    df.pipe(add_column, col_name="D", value=10)
      .pipe(multiply_column, col_name="A", factor=2)
)

row(df, df_res)

shape: (3, 3) A B C i64 i64 i64 1 4 7 2 5 8 3 6 9 
 shape: (3, 4) A B C D i64 i64 i64 i32 2 4 7 10 4 5 8 10 6 6 9 10

### Expr.pipe

表达式的 `pipe` 方法可以对表达式套用灵活的数据转换和自定义逻辑。下面定义一个计算二次式的函数 `quadratic()`，再用 `pipe` 把它套到多个表达式上。

`pipe()` 的第一个参数是处理 `Expr` 的函数，表达式会自动作为该函数的第一个参数传入。`pipe()` 其余的定位参数和关键字参数原样转给这个函数。

下面的代码用 `.pipe(quadratic, ...)` 做两件事。

  - 对 `A` 列计算 $ x^2 + 2 \cdot x + 3 $，列名加 `"_2"` 后缀。
  - 对 `B` 列与 `C` 列的和计算 $ x^2 + 2 \cdot x $，列名加 `"_q"` 后缀。

In [3]:
def quadratic(x, a=0, b=0, c=0, suffix='_2'):
    return (a * x ** 2 + b * x + c).name.suffix(suffix)

df_res = (
    df.with_columns(
        pl.col("A").pipe(quadratic, 1, 2, 3),
        (pl.col("B") + pl.col("C")).pipe(quadratic, a=1, b=2, suffix='_q')
    )
)

row(df, df_res)

shape: (3, 3) A B C i64 i64 i64 1 4 7 2 5 8 3 6 9 
 shape: (3, 5) A B C literal_2 literal_q i64 i64 i64 i64 i64 1 4 7 6 143 2 5 8 11 195 3 6 9 18 255

## map_batches

`pl.map_batches` 用来把 DataFrame 的多列合在一起处理。它把指定的列交给用户定义函数算出结果，再作为新列加进表里。

```python
pl.map_batches(
    column_names, 
    f: Callable[[Sequence[Series]], Series | Any], 
    return_dtype: PolarsDataType | pl.DataTypeExpr | None = None
)
```

* `column_names`：交给用户函数的列名或表达式列表。
* `f`：处理数据的用户定义函数。它接收指定列的 `pl.Series` 列表，返回新的 `pl.Series` 或其它值。
* `return_dtype`：用户函数返回值中元素的数据类型。默认 `None`，此时自动推断。

下面的例子用 `a`、`b` 两列调用用户定义函数 `hypot`，把结果作为新列 `c` 加进 DataFrame。

In [4]:
df = pl.DataFrame(
    {
        "a": [3, 3, 3, 4],
        "b": [4, 12, 6, 7],
        "g": ['A', 'B', 'A', 'B']
    }
)

def hypot(args:list[pl.Series]):
    a, b = args
    return (a**2 + b**2)**0.5
    
df.with_columns(
    pl.map_batches(['a', 'b'], hypot, return_dtype=pl.Float64).alias('c')
)

a,b,g,c
i64,i64,str,f64
3,4,"""A""",5.0
3,12,"""B""",12.369317
3,6,"""A""",6.708204
4,7,"""B""",8.062258


`column_names` 参数也可以传表达式。下面的代码用「`a` 列加 1」和 `b` 列做计算。

In [5]:
df.with_columns(
    pl.map_batches([pl.col('a') + 1, 'b'], hypot).alias('c')

)

a,b,g,c
i64,i64,str,f64
3,4,"""A""",5.656854
3,12,"""B""",12.649111
3,6,"""A""",7.211103
4,7,"""B""",8.602325


除表达式和用户函数外，`pl.map_batches()` 还有下面几个参数。

* `return_dtype`（默认：`None`）：
  指定用户函数返回的 **Series 元素的数据类型**。
  不指定时由 Polars 推断输出类型。

* `is_elementwise`（默认：`False`）：
  用来告诉 Polars 用户函数是否 **逐元素（elementwise）** 处理。
  设为 `True` 后，不论有没有分组，输入数据都会整块交给用户函数。

* `returns_scalar`（默认：`False`）：
  用户函数 **返回标量值** 时指定。
  不指定的话，Polars 按返回 Series 的函数处理。

下面的 `f1()`、`f2()`、`f3()` 分别演示这些参数的作用。

* `f1()`：只做元素之间的计算，返回 Series
* `f2()`：不是元素之间的计算，但返回 Series
* `f3()`：返回的不是 Series，而是 **标量值**

In [6]:
import threading

lock = threading.Lock()

def f1(args):
    with lock:
        a, b = args
        print(f"{a.to_list()} {b.to_list()}")
        return a + b

def f2(args):
    with lock:
        a, b = args
        print(f"{a.to_list()} {b.to_list()}")
        return a - b.min()

def f3(args):
    with lock:
        a, b = args
        print(f"{a.to_list()} {b.to_list()}")
        return a.max() - b.min()

不指定 `return_dtype` 时，Polars 会用假输入（`[1, 1]`）先把函数跑一遍，以此推断输出类型。

In [7]:
df.select(
    pl.map_batches(['a', 'b'], f1)
);

[1, 1] [1, 1]
[3, 3, 3, 4] [4, 12, 6, 7]


指定 `return_dtype` 后，这次试跑就不需要了。

In [8]:
df.select(
    pl.map_batches(['a', 'b'], f1, return_dtype=pl.Int64)
);

[3, 3, 3, 4] [4, 12, 6, 7]


(map_batches_in_agg)=

在 `agg()` 上下文里，`is_elementwise` 的作用看得更清楚。为 `False` 时，各组数据 **按组** 交给用户函数。
结果总是对的，但函数会被调用多次，速度慢一些。

In [9]:
df.group_by('g', maintain_order=True).agg(
    pl.map_batches(['a', 'b'], f1, return_dtype=pl.Int64, is_elementwise=False).alias('f1'),
    pl.map_batches(['a', 'b'], f2, return_dtype=pl.Int64, is_elementwise=False).alias('f2'),
)

[3, 3] [4, 6]
[3, 3] [4, 6]
[3, 4] [12, 7]
[3, 4] [12, 7]


g,f1,f2
str,list[i64],list[i64]
"""A""","[7, 9]","[-1, -1]"
"""B""","[15, 11]","[-4, -3]"


为 `True` 时，和分组无关，**所有数据一次性** 交给用户函数。

* 像 `f1()` 这种「元素之间的计算」，结果正确
* 像 `f2()` 这种「不是逐元素计算」的函数，**结果会错**

In [10]:
df.group_by('g', maintain_order=True).agg(
    pl.map_batches(['a', 'b'], f1, return_dtype=pl.Int64, is_elementwise=True).alias('f1'),
    pl.map_batches(['a', 'b'], f2, return_dtype=pl.Int64, is_elementwise=True).alias('f2'),
)

[3, 3, 3, 4] [4, 12, 6, 7]
[3, 3, 3, 4] [4, 12, 6, 7]


g,f1,f2
str,list[i64],list[i64]
"""A""","[7, 9]","[-1, -1]"
"""B""","[15, 11]","[-1, 0]"


`f3()` 返回标量（一个数值），不指定 `returns_scalar=True` 就会报错。

In [11]:
df.group_by('g', maintain_order=True).agg(
    pl.map_batches(['a', 'b'], f3, return_dtype=pl.Int64, returns_scalar=True).alias('f3'),
)

[3, 3] [4, 6]
[3, 4] [12, 7]


g,f3
str,i64
"""A""",-1
"""B""",-3


表达式也有 `map_batches()` 方法。下面的例子用 `Expr.map_batches()` 把表达式的计算结果交给用户函数处理。两列的数据分两次传给 `square()`。

In [12]:
def square(s):
    return s**2

df.select(
    pl.col('a', 'b').map_batches(square)
)

a,b
i64,i64
9,16
9,144
9,36
16,49


也可以先用 `pl.struct()` 把多列合成一个结构体列，再用 `map_batches()` 套自定义函数。下面是处理多列的示例代码。

In [13]:
def square2(s):
    return s.struct.field('a')**2 + s.struct.field('b')**2

df.select(
    pl.struct('a', 'b').map_batches(square2)
)

a
i64
25
153
45
65


## map_groups

`pl.map_groups()` 把各组数据交给用户定义函数处理，再把结果汇总。
实现上 `map_groups()` 和 `map_batches()` 是同一套机制，但在 `agg()` 上下文里用 `map_groups()` 更能看出意图，代码也更好读。

下面的例子按组计算 `a` 列最大值与 `b` 列最大值的比。

In [14]:
df = pl.DataFrame(
    {
        "a": [3, 3, 3, 4],
        "b": [4, 12, 6, 7],
        "g": ['A', 'B', 'A', 'B']
    }
)

def ratio_max(args):
    a, b = args
    return a.max() / b.max()

df.group_by('g').agg(
    pl.map_groups(['a', 'b'], ratio_max, returns_scalar=True)
)

g,a
str,f64
"""A""",0.5
"""B""",0.333333


## map_elements

用 `Expr.map_elements()` 把表达式的每个值交给自定义函数，结果作为新列。下面的例子用 `f` 函数逐个处理取值。

In [15]:
def f(x):
    if x > 10:
        return 'large'
    elif x > 5:
        return 'middle'
    else:
        return 'small'

df.with_columns(
    pl.col.b.map_elements(f, return_dtype=pl.String).alias('b_category')
)

a,b,g,b_category
i64,i64,str,str
3,4,"""A""","""small"""
3,12,"""B""","""large"""
3,6,"""A""","""middle"""
4,7,"""B""","""middle"""


`Expr.map_elements()` 逐个处理取值，数据量大时会慢。因此只在向量化运算和 Polars 内置函数做不到时才用。上面的例子也可以改写成向量化的条件分支：

In [16]:
df.with_columns(
    pl.when(pl.col('b') > 10).then(pl.lit('large'))
      .when(pl.col('b') > 5).then(pl.lit('middle'))
      .otherwise(pl.lit('small'))
      .alias('b_category')
)

a,b,g,b_category
i64,i64,str,str
3,4,"""A""","""small"""
3,12,"""B""","""large"""
3,6,"""A""","""middle"""
4,7,"""B""","""middle"""


在 `agg()` 或 `over()` 上下文里用 `map_elements()`，传给用户函数的仍然是 **标量值**。

下面的例子中 `f` 对每个元素各调用一次，值依次传入。

In [17]:
def f(x):
    print("---")
    print(x)
    return x

g = df.group_by('g', maintain_order=True)
g.agg(pl.col('a').map_elements(f, return_dtype=pl.Int64))

---
3
---
3
---
3
---
4


g,a
str,list[i64]
"""A""","[3, 3]"
"""B""","[3, 4]"


想把整组数据交给用户函数，得用 `map_batches()`，或者 `.implode().map_elements()`。

要注意 `map_elements()` 的 `return_dtype` 和 `map_batches()` 含义略有不同。

* `map_batches()` 的 `return_dtype` 指定 **用户函数返回值的元素类型**。
* `map_elements()` 的 `return_dtype` 指定 **用户函数返回值本身的数据类型**。

比如 `.implode().map_elements()` 的返回值是 **Series**，所以 `return_dtype` 要给 `pl.List(...)`。

In [18]:
g.agg(pl.col('a').implode().map_elements(f, return_dtype=pl.List(pl.Int64)))

---
shape: (2,)
Series: '' [i64]
[
	3
	3
]
---
shape: (2,)
Series: '' [i64]
[
	3
	4
]


g,a
str,list[i64]
"""A""","[3, 3]"
"""B""","[3, 4]"


In [19]:
g.agg(pl.col('a').map_batches(f, return_dtype=pl.Int64))

---
shape: (2,)
Series: '' [i64]
[
	3
	4
]
---
shape: (2,)
Series: '' [i64]
[
	3
	3
]


g,a
str,list[i64]
"""A""","[3, 3]"
"""B""","[3, 4]"


## rolling_map

`pl.rolling_mean()` 这类移动窗口函数覆盖不到的情况，可以用 `rolling_map()` 把窗口内的数据交给自定义函数处理。下面的代码把每个窗口内的数据传给用户函数，结果作为新列。

:::{tip}
`pl.Series._s`：用来访问 Series 对象的内部数据，可以借此确认实际数据及其所在内存的 ID。
:::

`arguments` 列表记录每个窗口的数据和该 Series 对象的内存 ID。为了省内存，同一个 Series 对象会被反复传入，输出里能看到这一点。

In [20]:
df = pl.DataFrame(
    {
        "a": [2, 4, 0, 3, 1, 6]
    }
)

arguments = []
def f(s):
    arguments.append((id(s._s), s.to_list()))
    return s.mean()

df.select(
    pl.col('a').rolling_map(f, window_size=3, center=True, min_samples=1)
)

for id_, arg in arguments:
    print(id_, arg)

1977640039168 [2, 4]
1977640039168 [2, 4, 0]
1977640039168 [4, 0, 3]
1977640039168 [0, 3, 1]
1977640039168 [3, 1, 6]
1977640039168 [1, 6]


## ufunc

可以直接用 NumPy 的 `ufunc` 对列或表达式做运算。比如 `np.hypot` 计算两数平方和的平方根 $\sqrt{a^2+b^2}$，在 Polars 里这样写：

In [21]:
df = pl.DataFrame(
    {
        "a": [3, 3, 3, 4],
        "b": [4, 12, 6, 7],
        "g": ['A', 'B', 'A', 'B']
    }
)

df.with_columns(
    np.hypot(pl.col("a"), pl.col("b")).alias("c")
)

a,b,g,c
i64,i64,str,f64
3,4,"""A""",5.0
3,12,"""B""",12.369317
3,6,"""A""",6.708204
4,7,"""B""",8.062258


用 `np.hypot()` 对列做运算时，Polars 内部怎么处理？看看表达式本身就清楚了：它把 `a` 列转成结构体列，再把 `b` 列作为 "argument_1" 加进去，然后对这个结构体列调用 `python_udf()`，由它调 NumPy 的 `np.hypot()`。

In [22]:
np.hypot(pl.col("a"), pl.col("b"))

<Expr ['as_struct("a", col("b").alias(…'] at 0x1CC74837F20>

## 列名与字段名改写

**`Expr.name.map()`**

用用户函数改写列名。

**`Expr.name.map_fields()`**

用用户函数改写 Struct 的 field 名。

In [23]:
def rename(n):
    return n.upper() + n
    
df.select(pl.all().name.map(rename))

Aa,Bb,Gg
i64,i64,str
3,4,"""A"""
3,12,"""B"""
3,6,"""A"""
4,7,"""B"""


`DataFrame.map_rows()`

把每行作为 Tuple 交给用户函数。

In [24]:
def f(row):
    a, b, g = row
    return f"{g}:{a + b}", f"{g}:{a - b}"
    
df.map_rows(f).rename({"column_0": "add", "column_1": "sub"})

add,sub
str,str
"""A:7""","""A:-1"""
"""B:15""","""B:-9"""
"""A:9""","""A:-3"""
"""B:11""","""B:-3"""


需要按字典逐行处理时，用 `DataFrame.iter_rows()`。

In [25]:
for row in df.iter_rows(named=True):
    print(row)

{'a': 3, 'b': 4, 'g': 'A'}
{'a': 3, 'b': 12, 'g': 'B'}
{'a': 3, 'b': 6, 'g': 'A'}
{'a': 4, 'b': 7, 'g': 'B'}
